# House Price Prediction — phiên bản người mới

Notebook này là lần đầu mình thử ghép các mảnh kiến thức Machine Learning lại với nhau. Mục tiêu của mình khá đơn giản: đọc dữ liệu, tự nghĩ cách xử lý vài feature, rồi tự viết hồi quy tuyến tính bằng NumPy.

Mình không cố tìm mô hình tốt nhất. Mình chỉ muốn nhìn rõ dữ liệu đi qua từng bước và có thể tự giải thích lại.

## 1. Chuẩn bị môi trường

Phần xử lý bảng và vẽ hình được giao cho thư viện. Phần mô hình, gradient descent và metric sẽ tự viết bằng NumPy.

In [ ]:
import os
import platform
from pathlib import Path

# Đặt cache trong thư mục đang chạy để notebook dùng được cả trong sandbox.
CACHE_DIR = Path.cwd() / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 100)

print("Python:", platform.python_version())
print("NumPy :", np.__version__)
print("pandas:", pd.__version__)

## 2. Tải dữ liệu

Nếu chạy trên Google Colab và chưa có dữ liệu, cell dưới sẽ hiện nút upload. Hãy chọn cùng lúc train.csv, test.csv và sample_submission.csv.

Nếu chạy trong project này, cell sẽ tự tìm thư mục data.

In [ ]:
DATA_CANDIDATES = [
    Path("data"),
    Path("House-Price-Prediction/data"),
    Path("/content"),
    Path("/content/data"),
]

DATA_DIR = next(
    (path for path in DATA_CANDIDATES
     if (path / "train.csv").exists() and (path / "test.csv").exists()),
    None,
)

if DATA_DIR is None:
    try:
        from google.colab import files
        print("Hãy upload train.csv, test.csv và sample_submission.csv")
        files.upload()
        DATA_DIR = Path.cwd()
    except ImportError as exc:
        raise FileNotFoundError(
            "Không tìm thấy train.csv và test.csv. Hãy đặt chúng trong thư mục data."
        ) from exc

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_path = DATA_DIR / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path) if sample_path.exists() else None

assert "SalePrice" in train_df.columns
assert "SalePrice" not in test_df.columns
assert train_df.drop(columns="SalePrice").columns.equals(test_df.columns)

print("Đọc dữ liệu từ:", DATA_DIR.resolve())
print("Train:", train_df.shape)
print("Test :", test_df.shape)
display(train_df.head())

## 3. Xem feature nào đang thiếu dữ liệu

Mình chưa sửa gì cả. Trước tiên mình chỉ hỏi pandas: mỗi cột đang trống bao nhiêu ô?

In [ ]:
missing_count = train_df.isna().sum()
missing_table = pd.DataFrame({
    "Số ô trống": missing_count,
    "Phần trăm trống": missing_count / len(train_df) * 100,
})
missing_table = missing_table[missing_table["Số ô trống"] > 0]
missing_table = missing_table.sort_values("Số ô trống", ascending=False)

print("Có", len(missing_table), "feature bị thiếu dữ liệu trong train.csv")
display(missing_table.round(2))

ax = missing_table["Phần trăm trống"].sort_values().plot.barh(
    figsize=(9, 7), color="steelblue"
)
ax.set_title("Các feature bị thiếu dữ liệu")
ax.set_xlabel("Phần trăm trống")
plt.tight_layout()
plt.show()

## 4. Mình chọn feature bằng suy luận đơn giản

Mình đoán giá nhà thường liên quan đến chất lượng, diện tích, tuổi nhà, vị trí và garage. Vì vậy mình chọn 11 feature gốc dưới đây, thay vì đưa cả 79 feature vào rồi không giải thích nổi.

- OverallQual: chất lượng tổng thể cao thì nhà có lẽ đắt hơn.
- GrLivArea, LotArea và LotFrontage: nhà hoặc đất rộng thường có giá cao hơn.
- GarageCars: garage chứa được nhiều xe có vẻ là một điểm cộng.
- Fireplaces: lò sưởi có thể đại diện cho tiện nghi.
- MasVnrArea: diện tích ốp đá có thể liên quan đến độ hoàn thiện.
- Neighborhood: vị trí thường ảnh hưởng mạnh đến giá nhà.
- HouseStyle: kiểu nhà khác nhau có thể có giá khác nhau.
- KitchenQual: bếp tốt có thể làm nhà hấp dẫn hơn.
- GarageType: loại garage có thể cho biết garage tốt hay chỉ là chỗ để xe đơn giản.

Mình còn tự ghép thêm bốn feature: tổng diện tích, tổng số phòng tắm, tuổi nhà lúc bán và số năm từ lần sửa gần nhất. Đây là phỏng đoán chủ quan, không phải kết quả của một cuộc thử nghiệm chọn feature.

## 5. Chia 80% train và 20% kiểm tra

Mình chia dữ liệu trước khi tính mean hoặc mode. Như vậy 20% kiểm tra không lén nói đáp án cho phần xử lý dữ liệu.

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_df))
split_position = int(0.80 * len(train_df))

train_indices = indices[:split_position]
holdout_indices = indices[split_position:]

train_part = train_df.iloc[train_indices].copy()
holdout_part = train_df.iloc[holdout_indices].copy()
kaggle_test_part = test_df.copy()

y_train = train_part["SalePrice"].to_numpy(dtype=float)
y_holdout = holdout_part["SalePrice"].to_numpy(dtype=float)

print("80% để học      :", len(train_part), "căn nhà")
print("20% để kiểm tra :", len(holdout_part), "căn nhà")
print("Kaggle test.csv :", len(kaggle_test_part), "căn nhà")

## 6. Tạo bốn feature mới

Các phép cộng dưới đây chỉ dùng thông tin của chính căn nhà đó. Nếu một phần basement hoặc phòng tắm bị thiếu, mình tạm xem phần đó bằng 0.

In [ ]:
ORIGINAL_FEATURES = [
    "OverallQual", "GrLivArea", "GarageCars", "LotArea",
    "LotFrontage", "Fireplaces", "MasVnrArea",
    "Neighborhood", "HouseStyle", "KitchenQual", "GarageType",
]

ENGINEERED_FEATURES = [
    "TotalSF", "TotalBathrooms", "HouseAgeAtSale", "YearsSinceRemodel",
]

FINAL_FEATURES = ORIGINAL_FEATURES + ENGINEERED_FEATURES
CATEGORICAL_FEATURES = [
    "Neighborhood", "HouseStyle", "KitchenQual", "GarageType",
]

def make_features(df):
    result = df.copy()
    result["TotalSF"] = (
        result["TotalBsmtSF"].fillna(0)
        + result["1stFlrSF"].fillna(0)
        + result["2ndFlrSF"].fillna(0)
    )
    result["TotalBathrooms"] = (
        result["FullBath"].fillna(0)
        + 0.5 * result["HalfBath"].fillna(0)
        + result["BsmtFullBath"].fillna(0)
        + 0.5 * result["BsmtHalfBath"].fillna(0)
    )
    result["HouseAgeAtSale"] = result["YrSold"] - result["YearBuilt"]
    result["YearsSinceRemodel"] = result["YrSold"] - result["YearRemodAdd"]
    return result[FINAL_FEATURES]

train_features_raw = make_features(train_part)
holdout_features_raw = make_features(holdout_part)
kaggle_features_raw = make_features(kaggle_test_part)

print("Số feature mình chọn:", len(FINAL_FEATURES))
display(train_features_raw.head())

## 7. Điền dữ liệu thiếu cho 15 feature đã chọn

Cách suy luận của mình:

- MasVnrArea thiếu thì xem như không có phần ốp đá và điền 0.
- GarageCars thiếu thì xem như không có chỗ để xe và điền 0.
- GarageType thiếu thì điền None, nghĩa là không có garage.
- LotFrontage là một số đo thật, nên mình điền mean của 80% train.
- Nếu sau này một feature số khác bị thiếu, dùng mean; categorical khác bị thiếu, dùng mode của 80% train.

Đây chỉ là các giả định đơn giản đầu tiên của mình. Chúng có thể chưa phải lựa chọn tối ưu.

In [ ]:
ZERO_NUMERIC_FEATURES = {"GarageCars", "MasVnrArea"}
NONE_CATEGORICAL_FEATURES = {"GarageType"}
NUMERIC_FEATURES = [
    column for column in FINAL_FEATURES
    if column not in CATEGORICAL_FEATURES
]

# Chỉ học giá trị điền từ 80% train.
numeric_fill_values = {}
for column in NUMERIC_FEATURES:
    if column in ZERO_NUMERIC_FEATURES:
        numeric_fill_values[column] = 0.0
    else:
        numeric_fill_values[column] = float(train_features_raw[column].mean())

categorical_fill_values = {}
for column in CATEGORICAL_FEATURES:
    if column in NONE_CATEGORICAL_FEATURES:
        categorical_fill_values[column] = "None"
    else:
        categorical_fill_values[column] = str(
            train_features_raw[column].mode().iloc[0]
        )

fill_rules = pd.DataFrame({
    "Feature": list(numeric_fill_values) + list(categorical_fill_values),
    "Giá trị dùng để điền": (
        list(numeric_fill_values.values())
        + list(categorical_fill_values.values())
    ),
})
display(fill_rules)

In [ ]:
def fill_missing_values(df):
    result = df.copy()
    for column, value in numeric_fill_values.items():
        result[column] = pd.to_numeric(
            result[column], errors="coerce"
        ).fillna(value)
    for column, value in categorical_fill_values.items():
        result[column] = (
            result[column].astype("string").fillna(value).astype(str)
        )
    return result

train_features = fill_missing_values(train_features_raw)
holdout_features = fill_missing_values(holdout_features_raw)
kaggle_features = fill_missing_values(kaggle_features_raw)

assert not train_features.isna().any().any()
assert not holdout_features.isna().any().any()
assert not kaggle_features.isna().any().any()

print("Ô trống còn lại ở 80% train :", train_features.isna().sum().sum())
print("Ô trống còn lại ở 20% test  :", holdout_features.isna().sum().sum())
print("Ô trống còn lại ở test.csv  :", kaggle_features.isna().sum().sum())

## 8. Đổi categorical feature thành các cột 0 và 1

Hồi quy tuyến tính chỉ nhân được với số. pandas.get_dummies sẽ biến từng lựa chọn như Neighborhood thành nhiều công tắc 0/1.

Danh sách cột được lấy từ 80% train. Hai tập còn lại chỉ được sắp theo danh sách đó.

In [ ]:
X_train_table = pd.get_dummies(
    train_features, columns=CATEGORICAL_FEATURES, dtype=float
)
ENCODED_COLUMNS = X_train_table.columns.tolist()

def encode_like_train(df):
    encoded = pd.get_dummies(
        df, columns=CATEGORICAL_FEATURES, dtype=float
    )
    return encoded.reindex(columns=ENCODED_COLUMNS, fill_value=0.0)

X_holdout_table = encode_like_train(holdout_features)
X_kaggle_table = encode_like_train(kaggle_features)

assert X_train_table.columns.equals(X_holdout_table.columns)
assert X_train_table.columns.equals(X_kaggle_table.columns)

print("Trước one-hot:", len(FINAL_FEATURES), "feature")
print("Sau one-hot  :", X_train_table.shape[1], "cột số")
display(X_train_table.head())

## 9. Chuẩn hóa feature

LotArea có thể lớn hàng chục nghìn, trong khi GarageCars chỉ khoảng 0 đến 4. Nếu giữ nguyên, gradient của các feature có thang đo lớn dễ lấn át phần còn lại.

Mình đưa mỗi cột về gần mean 0 và độ lệch chuẩn 1. Mean và std vẫn chỉ được tính từ 80% train.

In [ ]:
def fit_standardizer(X_train):
    X_train = np.asarray(X_train, dtype=float)
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    return mean, std

def transform(X, mean, std):
    X = np.asarray(X, dtype=float)
    safe_std = np.where(std == 0, 1.0, std)
    result = (X - mean) / safe_std
    result[:, std == 0] = 0.0
    return result

X_train_raw = X_train_table.to_numpy(dtype=float)
X_holdout_raw = X_holdout_table.to_numpy(dtype=float)
X_kaggle_raw = X_kaggle_table.to_numpy(dtype=float)

feature_mean, feature_std = fit_standardizer(X_train_raw)
X_train = transform(X_train_raw, feature_mean, feature_std)
X_holdout = transform(X_holdout_raw, feature_mean, feature_std)
X_kaggle = transform(X_kaggle_raw, feature_mean, feature_std)

assert X_train.shape[1] == X_holdout.shape[1] == X_kaggle.shape[1]
assert np.isfinite(X_train).all()
assert np.isfinite(X_holdout).all()
assert np.isfinite(X_kaggle).all()

print("X_train shape  :", X_train.shape)
print("X_holdout shape:", X_holdout.shape)
print("X_kaggle shape :", X_kaggle.shape)
print("Mean vài cột đầu sau chuẩn hóa:", X_train.mean(axis=0)[:5].round(4))

## 10. Tự viết hồi quy tuyến tính bằng NumPy

Mô hình dự đoán theo công thức:

y_pred = X @ w + b

Ban đầu mọi weight và bias đều bằng 0. Mỗi epoch, mình tính sai số, gradient rồi dịch w và b một bước nhỏ theo hướng làm loss giảm.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    error = y_true - y_pred
    mae = np.mean(np.abs(error))
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    ss_res = np.sum(error ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1.0 - ss_res / ss_tot
    return float(mae), float(mse), float(rmse), float(r2)


class LinearRegressionGD:
    def __init__(self, learning_rate=0.003, epochs=5000):
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y, print_every=250):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        m, n = X.shape

        self.w_ = np.zeros(n, dtype=float)
        self.b_ = 0.0
        self.loss_history_ = []

        initial_error = X @ self.w_ + self.b_ - y
        initial_loss = np.mean(initial_error ** 2) / 2.0
        self.loss_history_.append(float(initial_loss))
        print(f"Epoch {0:4d}/{self.epochs} | loss = {initial_loss:,.2f}")

        for epoch in range(1, self.epochs + 1):
            y_pred = X @ self.w_ + self.b_
            error = y_pred - y

            dw = X.T @ error / m
            db = np.mean(error)

            self.w_ -= self.learning_rate * dw
            self.b_ -= self.learning_rate * db

            new_error = X @ self.w_ + self.b_ - y
            loss = np.mean(new_error ** 2) / 2.0
            self.loss_history_.append(float(loss))

            if epoch % print_every == 0 or epoch == self.epochs:
                print(f"Epoch {epoch:4d}/{self.epochs} | loss = {loss:,.2f}")

        self.loss_history_ = np.asarray(self.loss_history_)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w_ + self.b_

## 11. Bấm cell này để nhìn mô hình train

Epoch là số lần mô hình nhìn qua toàn bộ 80% train. Loss ở đây là MSE chia 2 nên số khá lớn vì mình đang học trực tiếp giá nhà theo USD.

In [ ]:
model = LinearRegressionGD(learning_rate=0.003, epochs=5000)
model.fit(X_train, y_train, print_every=250)

assert np.isfinite(model.loss_history_).all()
assert model.loss_history_[-1] < model.loss_history_[0]
print("\nTrain xong!")

## 12. Mảng trọng số mô hình học được

Mỗi cột sau one-hot có một weight. Vì feature đã được chuẩn hóa, weight dương lớn thường đẩy dự đoán lên và weight âm lớn thường kéo dự đoán xuống. Đây chỉ là cách mô hình đang sử dụng feature, không chứng minh nguyên nhân thật ngoài đời.

In [ ]:
print("Bias:", round(model.b_, 2))
print("Mảng weights đầy đủ:")
print(model.w_)

weight_table = pd.DataFrame({
    "Feature sau one-hot": ENCODED_COLUMNS,
    "Weight": model.w_,
})
weight_table["Độ lớn"] = weight_table["Weight"].abs()
weight_table = weight_table.sort_values("Độ lớn", ascending=False)

print("\n20 weight có độ lớn cao nhất:")
display(weight_table.head(20).drop(columns="Độ lớn"))

## 13. Đánh giá trên 20% dữ liệu mô hình chưa học

Baseline đơn giản nhất là luôn đoán bằng giá trung bình của 80% train. Mô hình của mình nên có RMSE thấp hơn baseline này.

In [ ]:
holdout_prediction = model.predict(X_holdout)
baseline_prediction = np.full_like(y_holdout, y_train.mean())

baseline_mae, baseline_mse, baseline_rmse, baseline_r2 = regression_metrics(
    y_holdout, baseline_prediction
)
model_mae, model_mse, model_rmse, model_r2 = regression_metrics(
    y_holdout, holdout_prediction
)

metric_table = pd.DataFrame([
    {
        "Mô hình": "Đoán bằng mean",
        "MAE": baseline_mae,
        "MSE": baseline_mse,
        "RMSE": baseline_rmse,
        "R2": baseline_r2,
    },
    {
        "Mô hình": "Linear Regression NumPy",
        "MAE": model_mae,
        "MSE": model_mse,
        "RMSE": model_rmse,
        "R2": model_r2,
    },
])

display(metric_table.round({"MAE": 2, "MSE": 2, "RMSE": 2, "R2": 4}))
assert model_rmse < baseline_rmse

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(model.loss_history_, color="royalblue")
axes[0].set_title("Loss giảm trong lúc train")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE / 2")
axes[0].grid(alpha=0.3)

axes[1].scatter(y_holdout, holdout_prediction, alpha=0.6, color="darkorange")
low = min(y_holdout.min(), holdout_prediction.min())
high = max(y_holdout.max(), holdout_prediction.max())
axes[1].plot([low, high], [low, high], "--", color="black")
axes[1].set_title("Giá thật và giá dự đoán")
axes[1].set_xlabel("Giá thật")
axes[1].set_ylabel("Giá dự đoán")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "Giá thật": y_holdout[:10],
    "Giá dự đoán": holdout_prediction[:10],
    "Sai lệch": holdout_prediction[:10] - y_holdout[:10],
})
display(comparison.round(2))

## 14. Dự đoán test.csv và tạo submission

Theo mục tiêu ban đầu, mình dùng nguyên mô hình đã học từ 80% train. Mình không fit lại bằng 100% dữ liệu.

Vì giá nhà không thể âm, dự đoán âm sẽ được chặn về 0 trước khi ghi file.

In [ ]:
kaggle_prediction = model.predict(X_kaggle)
kaggle_prediction = np.maximum(kaggle_prediction, 0.0)

submission = pd.DataFrame({
    "Id": test_df["Id"].to_numpy(),
    "SalePrice": kaggle_prediction,
})

assert len(submission) == len(test_df)
assert submission.columns.tolist() == ["Id", "SalePrice"]
assert submission["SalePrice"].notna().all()
assert (submission["SalePrice"] >= 0).all()
if sample_submission is not None:
    assert submission["Id"].equals(sample_submission["Id"])

default_output_dir = DATA_DIR.parent if DATA_DIR.name == "data" else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("HOUSE_PRICE_OUTPUT_DIR", default_output_dir))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission_beginner.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("Đã tạo:", SUBMISSION_PATH.resolve())
print("Số dòng:", len(submission))
display(submission.head())

try:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))
except ImportError:
    print("Không chạy trên Colab nên không tự tải file xuống.")

## 15. Điều mình rút ra

- Dữ liệu thực tế có nhiều ô trống và không phải ô trống nào cũng mang cùng một ý nghĩa.
- Categorical feature phải đổi thành số trước khi đưa vào hồi quy tuyến tính.
- Chuẩn hóa giúp gradient descent học dễ hơn khi các feature có đơn vị rất khác nhau.
- Loss giảm trên train chưa đủ; mình vẫn cần 20% chưa dùng để học nhằm đánh giá.
- Mô hình này còn đơn giản và các quyết định feature engineering chủ yếu dựa trên trực giác. Đó là điểm bắt đầu để mình hiểu quy trình trước khi học các cách làm phức tạp hơn.